## Pobranie danych

In [ ]:
import praw
import pandas as pd

# konfiguracja API
reddit = praw.Reddit(
client_id='PODAJ_ID',
client_secret='PODAJ_KLUCZ',
user_agent='USERNAME'
)

subreddit_name='CryptoCurrency'
data = []
subreddit = reddit.subreddit(subreddit_name)

for submission in subreddit.top(time_filter="year", limit=1000):
    data.append([submission.title])
    
    # pobranie wszystkich komentarzy do posta
    submission.comments.replace_more(limit=0)
    for comment in submission.comments.list():
        data.append([comment.body])

df = pd.DataFrame(data, columns=["text"])
df.to_csv("crypto_data.csv", index=False)

## Czyszczenie i filtrowanie danych

In [ ]:
import pandas as pd
import re
df = pd.read_csv("crypto_data.csv")

df = df.dropna().drop_duplicates()

# usunięcie linków, komentarzy moderatora, konwersja do małych liter
def clean_text(text):
    text = re.sub(r'http\S+', '', text)  
    return text.lower()

df = df[df['text'].str.contains('Your comment contained a link to telegram, which is hard blocked by reddit.')==False]
df['text'] = df['text'].apply(clean_text)

# komentarze pomiędzy 4 a 100 słów
df = df[df['text'].apply(lambda x: len(str(x).split()) >= 4)]
df = df[df['text'].apply(lambda x: len(str(x).split()) <= 100)]

lst = ['bitcoin', 'btc', 'eth', 'ethereum', 'sol', 'solana', 'xrp', 'ripple',
       'bnb', 'binance', 'ada', 'cardano', 'bch', 'usdt', 'tether',
       'monero', 'shib', 'doge', 'litecoin', 'ltc']

# regex dla całych słów
pattern = r'\b(' + '|'.join(re.escape(word) for word in lst) + r')\b'

# zapisanie komentarzy o jednej kryptowalucie
df['matches'] = df['text'].apply(lambda x: len(re.findall(pattern, x)))
df = df[df['matches'] == 1]
df = df.drop('matches', axis=1)
df.to_csv("clean_crypto_data.csv", index=False)

## Etykietowanie danych

In [ ]:
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import torch

crypto_model_name = "ElKulako/cryptobert"
crypto_fin_model_name= "kk08/CryptoBERT"

crypto_tokenizer = AutoTokenizer.from_pretrained(crypto_model_name)
crypto_fin_tokenizer = AutoTokenizer.from_pretrained(crypto_fin_model_name)
crypto_fin_model = AutoModelForSequenceClassification.from_pretrained(crypto_fin_model_name, num_labels=2)
crypto_model = AutoModelForSequenceClassification.from_pretrained(crypto_model_name, num_labels=3)

vader_analyzer = SentimentIntensityAnalyzer()

def classify_sentiment(text):
    results = []

    # VADER
    vader_score = vader_analyzer.polarity_scores(text)["compound"]
    if vader_score > 0.05:
        results.append("positive")
    elif vader_score < -0.05:
        results.append("negative")
    else:
        results.append("neutral")

    # kk08/CryptoBERT
    tokens = crypto_fin_tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    logits = crypto_fin_model(**tokens).logits
    fin_pred = ["negative", "positive"][torch.argmax(logits, dim=1).item()]
    results.append(fin_pred)

    # ElKulako/CryptoBERT
    tokens = crypto_tokenizer(text, return_tensors="pt", truncation=True, padding=True,max_length=512)
    logits = crypto_model(**tokens).logits
    crypto_pred = torch.argmax(logits, dim=1).item()
    results.append(["negative", "neutral", "positive"][crypto_pred])

    # głosowanie większościowe
    if len(set(results)) == 3:
      final_label = "neutral"
    else:
      final_label = max(set(results), key=results.count)

    return final_label, results  # lista wyników poszczególnych modeli

df_unlabeled = pd.read_csv("clean_crypto_data.csv")
df_unlabeled["pseudo_label"], df_unlabeled["model_votes"] = zip(*df_unlabeled["text"].apply(classify_sentiment))

df_unlabeled.to_csv("pseudolabeled_crypto_data.csv", index=False)

print(df_unlabeled.head())

## Augmentacja danych

In [ ]:
import pandas as pd
import nltk
import nlpaug.augmenter.word as naw

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')
augmenter = naw.SynonymAug(aug_src='wordnet', aug_max=3)

df = pd.read_csv("pseudolabeled_crypto_data.csv")
df= df.drop("model_votes", axis=1)

def augment_texts(texts, label, n=3):
    augmented_texts = []
    for text in texts:
        try:
            augmented_samples = augmenter.augment(text, n=n)
            for sample in augmented_samples:
                augmented_texts.append({"text": sample, "pseudo_label": label})
        except Exception as e:
            print(f"błąd: {text[:30]}... ({e})")
    return pd.DataFrame(augmented_texts)

# powielanie pozytywnych i negatywnych komentarzy 3 razy
df_positive = df[df["pseudo_label"] == "positive"]
df_negative = df[df["pseudo_label"] == "negative"]

df_pos_aug = augment_texts(df_positive["text"], "positive", n=3)
df_neg_aug = augment_texts(df_negative["text"], "negative", n=3)

# połączenie z oryginalnym zbiorem i wymieszanie kolejności
df_augmented = pd.concat([df, df_pos_aug, df_neg_aug], ignore_index=True)
df_augmented = df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)
df_augmented.to_csv("final_crypto_data.csv", index=False)

print(df_augmented["pseudo_label"].value_counts())

## Dostrajanie z poszukiwaniem hiperparametrów

In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
import torch
import wandb

wandb.init(project="cryptobert-param_search", mode="online")

df = pd.read_csv("final_crypto_data.csv")  
df = df.dropna(subset=["text", "pseudo_label"])

label_map = {
    "negative": 0,
    "neutral": 1,  
    "positive": 2
}

df["label"] = df["pseudo_label"].map(label_map).astype(int)

dataset = Dataset.from_pandas(df[["text", "label"]])
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

model_name = "ElKulako/cryptobert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = torch.argmax(torch.tensor(logits), axis=1).numpy()
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}

def model_init():
    return AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

def train(config=None):
    with wandb.init(config=config):
        config = wandb.config

        training_args = TrainingArguments(
            output_dir="./results",
            eval_strategy="epoch",
            logging_strategy="epoch",
            save_strategy="epoch",
            learning_rate=config.learning_rate,
            per_device_train_batch_size=config.batch_size,
            per_device_eval_batch_size=32,
            num_train_epochs=config.epochs,
            weight_decay=config.weight_decay,
            lr_scheduler_type=config.lr_scheduler,
            report_to="wandb",
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss", 
            greater_is_better=False,
            save_total_limit=1,
        )

        trainer = Trainer(
            model_init=model_init,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            tokenizer=tokenizer,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        )

        trainer.train()
        trainer.save_model(f"./finetuned_cryptobert_{wandb.run.name}")

#  konfiguracja hiperparametrów
sweep_config = {
    "method": "bayes",  
    "metric": {"name": "eval/accuracy", "goal": "maximize"},
    "parameters": {
        "learning_rate": {"min": 1e-6, "max": 5e-5},
        "batch_size": {"values": [16, 32, 64]},
        "epochs": {"values": [5]},
        "weight_decay": {"min": 0.0, "max": 0.1},
        "lr_scheduler": {"values": ["linear", "cosine"]},
    },
}

sweep_id = wandb.sweep(sweep_config, project="cryptobert-final")
wandb.agent(sweep_id, function=train, count=9)  # count = liczba modeli

## Macierz pomyłek dla najlepszego modelu

In [ ]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("final_crypto_data.csv")
df = df.dropna(subset=["text", "pseudo_label"])

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["pseudo_label"])

dataset = Dataset.from_pandas(df[["text", "label"]])
dataset = dataset.train_test_split(test_size=0.1, seed= 42)
eval_dataset = dataset["test"]

# ścieżka do modelu
model_path = "./finetuned_cryptobert_rich-sweep-2/"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=256)

eval_dataset = eval_dataset.map(tokenize_function, batched=True)
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

trainer = Trainer(model=model, tokenizer=tokenizer)
predictions = trainer.predict(eval_dataset)

y_true = predictions.label_ids
y_pred = torch.argmax(torch.tensor(predictions.predictions), dim=1).numpy()

true_labels = label_encoder.inverse_transform(y_true)
pred_labels = label_encoder.inverse_transform(y_pred)

results_df = pd.DataFrame({
    "text": eval_dataset["text"],  
    "true_label": true_labels,    
    "pred_label": pred_labels,   
})

# zapis do pliku nieprawidłowo sklasyfikowanych danych
misclassified_df = results_df[results_df["true_label"] != results_df["pred_label"]]
misclassified_df.to_csv("misclassified_samples.csv", index=False, encoding="utf-8")

# macierz pomyłek
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(cmap="Blues", values_format="d")
plt.title("Macierz pomyłek - model-2")
plt.xlabel("Przewidziana etykieta")
plt.ylabel("Rzeczywista etykieta") 
plt.show()

print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))